# 01 · Data preparation and dataset analysis (BraTS 2021)

This notebook:
1. downloads BraTS 2021 Task 1 from Kaggle **once** and keeps a permanent copy on Google Drive,
2. extracts it on the fast local Colab disk,
3. crops every patient to the brain and stores it compactly (lossless) in **shards** on Drive,
4. splits **patients** (not slices) into train / val / test and saves the split,
5. draws the dataset-analysis graphs.

**Runtime:** use a **CPU** runtime (Runtime → Change runtime type → CPU). It has the most disk space and no GPU is needed.

**Safe to re-run:** every finished shard is marked on Drive. If Colab disconnects, open the notebook again and *Run all*; finished work is skipped.

**Settings:** the defaults below are the final full run (1,251 patients, about 1–2 hours). `QUICK_RUN = True` runs a
40-patient smoke test instead; its data goes to `data_quick/` and never mixes with the real data.

In [ ]:
# ======================= SETTINGS - edit these ===========================
PROJECT_DIR = '/content/drive/MyDrive/BrainTumor'   # project folder on Google Drive
WORK_DIR    = '/content/work'                       # fast local disk of the Colab machine (temporary)

QUICK_RUN     = False    # False = full dataset (1,251 patients). True = smoke test on QUICK_N_CASES patients
QUICK_N_CASES = 40

KAGGLE_DATASET  = 'dschettler8845/brats-2021-task1'
SHARD_SIZE      = 100                  # patients per shard (the unit that is saved and resumed)
SPLIT           = (0.70, 0.15, 0.15)   # train / val / test fractions, by PATIENT
SEED            = 42
N_WORKERS       = None                 # None = use every CPU core
FORCE_NEW_SPLIT = False                # keep False: the split must not change once models are trained on it

In [ ]:
# ---- Setup: mount Google Drive, install missing packages, import the project code ----
import os
import sys

# Only used when testing outside Colab; on Colab these three lines change nothing.
PROJECT_DIR = os.environ.get('BT_PROJECT_DIR', PROJECT_DIR)
WORK_DIR = os.environ.get('BT_WORK_DIR', WORK_DIR)
QUICK_RUN = os.environ.get('BT_QUICK_RUN', '1' if QUICK_RUN else '0') == '1'

if PROJECT_DIR.startswith('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')

CODE_DIR = os.path.join(PROJECT_DIR, 'code')
if not os.path.isfile(os.path.join(CODE_DIR, 'brats', '__init__.py')):
    raise FileNotFoundError(f'The "brats" folder was not found in {CODE_DIR}.\n'
                            'Upload everything from the "Code" folder to that Drive folder first (see README).')
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

from brats.notebook import install_missing
install_missing({'nibabel': 'nibabel', 'sklearn': 'scikit-learn'})

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from brats import config, dataset, env, fileio, plots, raw_data

paths = env.ProjectPaths(PROJECT_DIR, WORK_DIR, quick=QUICK_RUN).make()
env.print_header(f'QUICK RUN ({QUICK_N_CASES} patients) - a smoke test, not real results' if QUICK_RUN
                 else 'FULL DATASET RUN')
print(paths)
env.print_environment(env.save_environment(paths.data_logs))
FIG = paths.data_figures

## Step 1 · Get the raw dataset
If all shards are already on Drive, the download and extraction are skipped.
Otherwise the notebook uses the copy in `data/raw/` on Drive, or downloads from Kaggle and saves a copy there.

Kaggle login (only needed once): add the Colab secrets `KAGGLE_USERNAME` and `KAGGLE_KEY` (key icon in the left sidebar), **or** put your `kaggle.json` at `MyDrive/BrainTumor/kaggle.json`.

In [ ]:
case_list_path = paths.meta / 'case_list.csv'
case_list = pd.read_csv(case_list_path) if case_list_path.is_file() else None
preprocessing_done = dataset.all_shards_done(paths, case_list)
print('Preprocessing already complete on Drive:', preprocessing_done)

archive_sources = []
if not preprocessing_done:
    archives = raw_data.list_archives(paths.raw)
    if archives:
        print('Using the dataset already saved on Drive:')
    else:
        local_archives = raw_data.list_archives(paths.work_download)
        if not local_archives:
            fileio.require_disk(paths.work, 32, 'downloading and extracting the dataset')
            print('Kaggle credentials found in:', raw_data.setup_kaggle_credentials(PROJECT_DIR))
            local_archives = raw_data.kaggle_download(KAGGLE_DATASET, paths.work_download)
        print('Saving a permanent copy of the raw dataset to Drive (only happens once)...')
        for archive in local_archives:
            fileio.copy_verified(archive, paths.raw / archive.name, progress=True)
        archives = raw_data.list_archives(paths.raw)
    for archive in archives:
        local_copy = paths.work_download / archive.name
        use_local = local_copy.is_file() and local_copy.stat().st_size == archive.stat().st_size
        archive_sources.append(local_copy if use_local else archive)
        print(f'  {archive.name:40s} {archive.stat().st_size / 1e9:6.2f} GB   '
              f'(reading from {"local disk" if use_local else "Drive"})')

## Step 2 · Extract the MRI files
Every file is written as `<case>/<case>_<modality>.nii.gz` on the local disk, whatever the archive layout is.

In [ ]:
complete_cases = {}
if not preprocessing_done:
    total_gb = sum(source.stat().st_size for source in archive_sources) / 1e9
    need_gb = (QUICK_N_CASES * 0.02 + 1) if QUICK_RUN else total_gb * 1.1 + 2
    fileio.require_disk(paths.work, need_gb, 'extracting the MRI files')
    wanted = case_list['case_id'].tolist() if case_list is not None else None
    case_files = raw_data.extract_brats_archives(
        archive_sources, paths.work_raw_cases, wanted_ids=wanted,
        max_cases=QUICK_N_CASES if (QUICK_RUN and wanted is None) else None)
    complete_cases, incomplete_cases = raw_data.check_case_files(case_files)
    print(f'{len(complete_cases)} complete patients, {len(incomplete_cases)} incomplete')
    if incomplete_cases:
        incomplete_df = pd.DataFrame({'case_id': list(incomplete_cases), 'problem': list(incomplete_cases.values())})
        fileio.save_csv(incomplete_df, paths.meta / 'incomplete_cases.csv')
        display(incomplete_df.head(20))
    if not complete_cases:
        raise RuntimeError('No complete patients were found in the archive(s). Check the files in data/raw/.')
    if case_list is None:
        case_list = dataset.make_case_list(complete_cases.keys(), SHARD_SIZE)
        fileio.save_csv(case_list, case_list_path)
        print(f'Case list saved: {len(case_list)} patients in {case_list["shard"].nunique()} shards')
    else:
        not_found = sorted(set(case_list['case_id']) - set(complete_cases))
        if not_found:
            print(f'WARNING: {len(not_found)} patients of the saved case list were not found: {not_found[:5]} ...')
    # The downloaded archive on the local disk is no longer needed (a copy is on Drive).
    for archive in raw_data.list_archives(paths.work_download):
        drive_copy = paths.raw / archive.name
        if drive_copy.is_file() and drive_copy.stat().st_size == archive.stat().st_size:
            archive.unlink()
            print(f'Freed local disk: removed {archive.name} (kept on Drive)')
else:
    print('Skipped - preprocessing is already complete.')

## Step 3 · Preprocess (crop to the brain, store compactly, collect statistics)
For each patient: the 4 modalities are cropped to the brain bounding box and stored as integers exactly as in the original files
(lossless, compressed). The z-score normalization parameters are stored alongside and applied on the fly during training,
so training and the final app normalize in exactly the same way.

Each shard of 100 patients is saved to Drive and marked as done; a disconnect loses at most one shard.

In [ ]:
if not preprocessing_done:
    fileio.require_disk(paths.work, 4, 'temporary shard files')
    dataset.preprocess_dataset(complete_cases, case_list, paths, n_workers=N_WORKERS,
                               preview_every=2 if QUICK_RUN else 20)
    if not dataset.all_shards_done(paths, case_list):
        raise RuntimeError('Some shards did not finish - run this cell again.')
else:
    print('Skipped - preprocessing is already complete.')

## Step 4 · Merge statistics and split patients into train / val / test
The split is made **once** and saved to `data/meta/split.csv`. Both models use exactly this split,
and slices of one patient never appear in two splits.

In [ ]:
cases, slices = dataset.merge_shard_statistics(paths)
failed = cases[cases['status'] != 'ok']
if len(failed):
    print(f'{len(failed)} patients failed preprocessing and are excluded:')
    display(failed[['case_id', 'error']])

split_path = paths.meta / 'split.csv'
if split_path.is_file() and not FORCE_NEW_SPLIT:
    split = pd.read_csv(split_path)
    print('Re-using the existing split from', split_path)
    unsplit = set(cases.loc[cases['status'] == 'ok', 'case_id']) - set(split['case_id'])
    if unsplit:
        print(f'WARNING: {len(unsplit)} patients are not in the saved split and will not be used.')
else:
    split = dataset.make_patient_split(cases, SPLIT, SEED)
    fileio.save_csv(split, split_path)
    print('New split saved to', split_path)

cases = cases.merge(split, on='case_id', how='left')
ok = cases[cases['status'] == 'ok']                 # the usable patients, with their split
slices = slices.merge(split, on='case_id', how='left')

summary = {
    'quick_run': QUICK_RUN, 'patients_ok': int(len(ok)), 'patients_failed': int(len(failed)),
    'patients_per_split': ok['split'].value_counts().to_dict(),
    'et_absent_patients': int((~ok['et_present'].astype(bool)).sum()),
    'stored_dtypes': ok['stored_dtype'].value_counts().to_dict(),
    'voxel_spacings': ok.groupby(['sp_x', 'sp_y', 'sp_z']).size().rename('n').reset_index().to_dict('records'),
    'shapes': ok.groupby(['shape_x', 'shape_y', 'shape_z']).size().rename('n').reset_index().to_dict('records'),
    'seg_voxels_outside_crop_total': int(ok['seg_outside_crop'].sum()),
    'wt_volume_cm3_median': float(ok['vol_wt_cm3'].median()),
    'preprocessed_size_gb': float(ok['npz_mb'].sum() / 1000),
    'slices_total': int(len(slices)),
}
fileio.save_json(summary, paths.meta / 'dataset_summary.json')
for key, value in summary.items():
    print(f'{key:32s} {value}')
assert summary['seg_voxels_outside_crop_total'] == 0, 'Some tumor voxels fall outside the brain crop!'

## Step 5 · Dataset analysis graphs
Every graph is saved to `data/figures/` as PNG + PDF, with a CSV of the numbers behind it.

In [ ]:
stats = dataset.load_dataset_stats(paths)
previews = dataset.load_previews(paths)
modalities = list(config.MODALITIES)
preview_ids = sorted(previews)
print(f'{len(preview_ids)} patients have preview slices saved')

# 1. Example patients: the 4 modalities and the expert label map
shown_ids = preview_ids[:4]
fig, axes = plt.subplots(len(shown_ids), 5, figsize=(12, 2.5 * len(shown_ids)), squeeze=False)
for row, case_id in enumerate(shown_ids):
    preview = previews[case_id]
    for col, modality in enumerate(modalities):
        plots.show_gray(axes[row, col], plots.view2d(preview['raw_axial'][col]),
                        plots.MOD_TITLES[modality] if row == 0 else None)
    plots.show_labels(axes[row, 4], plots.view2d(preview['raw_axial'][0]), plots.view2d(preview['seg_axial']),
                      'FLAIR + expert labels' if row == 0 else None)
    axes[row, 0].text(-0.08, 0.5, case_id, transform=axes[row, 0].transAxes, rotation=90, va='center', ha='right',
                      fontsize=8)
plots.label_legend(fig)
fig.suptitle('Example patients (axial slice with the largest tumor area)', y=1.0)
fig.tight_layout(rect=(0, 0.04, 1, 1))
plots.save_fig(fig, FIG, '01_sample_patients')

In [ ]:
# 2. What normalization does to one patient
case_id = shown_ids[0]
preview = previews[case_id]
fig, axes = plt.subplots(2, 4, figsize=(11, 5.6))
for col, modality in enumerate(modalities):
    plots.show_gray(axes[0, col], plots.view2d(preview['raw_axial'][col]), f'{plots.MOD_TITLES[modality]} - raw')
    axes[1, col].imshow(plots.view2d(preview['norm_axial'][col]), cmap='gray', vmin=-3, vmax=3)
    axes[1, col].set_axis_off()
    axes[1, col].set_title(f'{plots.MOD_TITLES[modality]} - z-scored', fontsize=9)
fig.suptitle(f'Before and after intensity normalization ({case_id})', y=1.0)
fig.tight_layout()
plots.save_fig(fig, FIG, '02_normalization_before_after')

In [ ]:
# 3. Three orthogonal views through the tumor
shown_ids = preview_ids[:3]
fig, axes = plt.subplots(len(shown_ids), 3, figsize=(9, 3.1 * len(shown_ids)), squeeze=False)
for row, case_id in enumerate(shown_ids):
    preview = previews[case_id]
    plots.show_labels(axes[row, 0], plots.view2d(preview['raw_axial'][0]), plots.view2d(preview['seg_axial']),
                      'Axial' if row == 0 else None)
    plots.show_labels(axes[row, 1], plots.view_vertical(preview['flair_coronal']),
                      plots.view_vertical(preview['seg_coronal']), 'Coronal' if row == 0 else None)
    plots.show_labels(axes[row, 2], plots.view_vertical(preview['flair_sagittal']),
                      plots.view_vertical(preview['seg_sagittal']), 'Sagittal' if row == 0 else None)
plots.label_legend(fig)
fig.suptitle('FLAIR with expert labels in three views', y=1.0)
fig.tight_layout(rect=(0, 0.04, 1, 1))
plots.save_fig(fig, FIG, '03_three_views')

In [ ]:
# 4. Tumor volumes per region
volume_columns = [('vol_wt_cm3', 'Whole tumor (WT)'), ('vol_tc_cm3', 'Tumor core (TC)'),
                  ('vol_et_cm3', 'Enhancing tumor (ET)'), ('vol_ncr_cm3', 'Necrotic core (NCR)'), ('vol_ed_cm3', 'Edema (ED)')]
fig, axes = plt.subplots(1, 5, figsize=(18, 3.4))
for i, (ax, (column, title)) in enumerate(zip(axes, volume_columns)):
    plots.hist_panel(ax, ok[column], bins=30, color=plots.PALETTE[i], title=title, xlabel='volume (cm³)')
fig.suptitle('Tumor volume per patient', y=1.04)
fig.tight_layout()
plots.save_fig(fig, FIG, '04_tumor_volumes', data=ok[['case_id'] + [column for column, _ in volume_columns]])

In [ ]:
# 5. Tumor size relative to the brain, and brain volume
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
plots.hist_panel(axes[0], ok['wt_brain_pct'], bins=30, title='Whole tumor as % of brain volume', xlabel='% of brain')
plots.hist_panel(axes[1], ok['brain_cm3'], bins=30, color=plots.PALETTE[1], title='Brain volume', xlabel='cm³')
fig.tight_layout()
plots.save_fig(fig, FIG, '05_tumor_share_and_brain_volume', data=ok[['case_id', 'wt_brain_pct', 'brain_cm3']])

In [ ]:
# 6. Composition of the tumor: share of necrotic core / edema / enhancing tumor
whole_tumor = ok['vol_wt_cm3'].clip(lower=1e-9)
composition = pd.DataFrame({'case_id': ok['case_id'], 'NCR': 100 * ok['vol_ncr_cm3'] / whole_tumor,
                            'ED': 100 * ok['vol_ed_cm3'] / whole_tumor, 'ET': 100 * ok['vol_et_cm3'] / whole_tumor})
fig, ax = plt.subplots(figsize=(6, 4))
plots.box_panel(ax, [composition['NCR'], composition['ED'], composition['ET']], ['Necrotic core', 'Edema', 'Enhancing'],
                colors=[plots.LABEL_COLORS[1], plots.LABEL_COLORS[2], plots.LABEL_COLORS[3]],
                ylabel='% of the whole tumor', title='What each tumor is made of')
fig.tight_layout()
plots.save_fig(fig, FIG, '06_tumor_composition', data=composition)

In [ ]:
# 7. Patients without enhancing tumor (important: ET Dice is 0 or 1 for them)
et_absent_pct = (ok.groupby('split')['et_present'].agg(lambda s: 100 * (1 - s.astype(bool).mean()))
                 .reindex(['train', 'val', 'test']))
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
counts = ok['et_present'].astype(bool).value_counts().reindex([True, False]).fillna(0)
plots.bar_panel(axes[0], ['ET present', 'ET absent'], counts.values, colors=[plots.PALETTE[2], plots.PALETTE[7]],
                title='Patients with / without enhancing tumor', ylabel='patients')
plots.bar_panel(axes[1], list(et_absent_pct.index), et_absent_pct.values,
                colors=[plots.SPLIT_COLORS[s] for s in et_absent_pct.index],
                title='% of patients without ET, per split', ylabel='%', fmt='{:.1f}')
fig.tight_layout()
plots.save_fig(fig, FIG, '07_enhancing_tumor_presence', data=et_absent_pct.rename('pct_et_absent').reset_index())

In [ ]:
# 8. Slice-level class balance for the detection model (same rule as notebook 02)
MIN_BRAIN_PIXELS, POS_MIN_PIXELS = 500, 1
eligible = slices[slices['brain_px'] >= MIN_BRAIN_PIXELS].copy()
eligible['tumor'] = eligible['wt_px'] >= POS_MIN_PIXELS
balance = eligible.groupby(['split', 'tumor']).size().unstack(fill_value=0).reindex(['train', 'val', 'test'])
fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(len(balance))
for j, (has_tumor, label, color) in enumerate([(False, 'no tumor', plots.PALETTE[0]), (True, 'tumor', plots.PALETTE[1])]):
    values = balance[has_tumor].values if has_tumor in balance else np.zeros(len(balance))
    bars = ax.bar(x + (j - 0.5) * 0.38, values, width=0.36, color=color, label=label, edgecolor='white')
    for bar, value in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, value, f'{int(value):,}', ha='center', va='bottom', fontsize=8,
                color=plots.INK2)
ax.set_xticks(x)
ax.set_xticklabels(balance.index)
ax.set_ylabel('axial slices')
ax.set_title(f'Slices with / without tumor (slices with ≥{MIN_BRAIN_PIXELS} brain pixels)')
ax.legend()
fig.tight_layout()
plots.save_fig(fig, FIG, '08_slice_class_balance', data=balance.reset_index())

In [ ]:
# 9. Where along the head (slice position) tumors appear
n_patients = ok['case_id'].nunique()
by_position = slices.groupby('z_orig').agg(tumor=('wt_px', lambda s: (s > 0).sum()),
                                           brain=('brain_px', lambda s: (s > 0).sum()))
by_position = by_position.reindex(range(int(ok['shape_z'].max())), fill_value=0)
fig, ax = plt.subplots(figsize=(8, 3.8))
ax.plot(by_position.index, 100 * by_position['brain'] / n_patients, label='brain visible', color=plots.PALETTE[0])
ax.plot(by_position.index, 100 * by_position['tumor'] / n_patients, label='tumor visible', color=plots.PALETTE[1])
ax.set_xlabel('axial slice index (0 = bottom of the volume)')
ax.set_ylabel('% of patients')
ax.set_title('How often each slice position contains brain / tumor')
ax.legend()
fig.tight_layout()
plots.save_fig(fig, FIG, '09_tumor_by_slice_position', data=by_position.reset_index())

In [ ]:
# 10. Tumor frequency map: how often each location is tumor, across all patients
n_freq = max(int(stats['n_freq']), 1)
tumor_freq = stats['freq_wt'] / n_freq
brain_freq = stats['freq_brain'] / n_freq
cx, cy, cz = (int(v) for v in np.array(tumor_freq.shape) // 2)
views = [('Axial (mean over slices)', plots.view2d(tumor_freq.mean(2)), plots.view2d(brain_freq.mean(2))),
         ('Coronal (mean)', plots.view_vertical(tumor_freq.mean(1)), plots.view_vertical(brain_freq.mean(1))),
         ('Sagittal (mean)', plots.view_vertical(tumor_freq.mean(0)), plots.view_vertical(brain_freq.mean(0))),
         ('Axial (centre slice)', plots.view2d(tumor_freq[:, :, cz]), plots.view2d(brain_freq[:, :, cz])),
         ('Coronal (centre slice)', plots.view_vertical(tumor_freq[:, cy, :]), plots.view_vertical(brain_freq[:, cy, :])),
         ('Sagittal (centre slice)', plots.view_vertical(tumor_freq[cx, :, :]), plots.view_vertical(brain_freq[cx, :, :]))]
fig, axes = plt.subplots(2, 3, figsize=(12, 7.5))
for ax, (title, tumor_view, brain_view) in zip(axes.ravel(), views):
    ax.imshow(brain_view, cmap='gray', vmin=0, vmax=max(brain_view.max(), 1e-6))
    image = ax.imshow(np.ma.masked_less_equal(tumor_view, 0), cmap=plots.SEQ_CMAP, alpha=0.85, vmin=0,
                      vmax=max(tumor_view.max(), 1e-6))
    ax.set_title(title, fontsize=9)
    ax.set_axis_off()
fig.colorbar(image, ax=axes.ravel().tolist(), shrink=0.6, label='fraction of patients with tumor here')
fig.suptitle(f'Tumor location frequency over {n_freq} patients', y=0.98)
plots.save_fig(fig, FIG, '10_tumor_frequency_map')
np.save(FIG / '10_tumor_frequency_map.npy', tumor_freq.astype(np.float32))

In [ ]:
# 11. Tumor centre positions (world coordinates, mm)
centres = ok[['wt_centroid_x_mm', 'wt_centroid_y_mm', 'wt_centroid_z_mm']].dropna()
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
projections = [('wt_centroid_x_mm', 'wt_centroid_y_mm', 'From above (x: left→right, y: back→front)'),
               ('wt_centroid_x_mm', 'wt_centroid_z_mm', 'From the front (x, z: down→up)'),
               ('wt_centroid_y_mm', 'wt_centroid_z_mm', 'From the side (y, z)')]
for ax, (x_column, y_column, title) in zip(axes, projections):
    ax.scatter(centres[x_column], centres[y_column], s=8, alpha=0.6, color=plots.PALETTE[0], linewidths=0)
    ax.set_xlabel(x_column.replace('wt_centroid_', '').replace('_mm', ' (mm)'))
    ax.set_ylabel(y_column.replace('wt_centroid_', '').replace('_mm', ' (mm)'))
    ax.set_title(title, fontsize=9)
    ax.set_aspect('equal', adjustable='datalim')
fig.suptitle('Whole-tumor centre of every patient', y=1.02)
fig.tight_layout()
plots.save_fig(fig, FIG, '11_tumor_centre_positions',
               data=ok[['case_id', 'wt_centroid_x_mm', 'wt_centroid_y_mm', 'wt_centroid_z_mm']])

In [ ]:
# 12. Which side of the brain
fig, axes = plt.subplots(1, 3, figsize=(11, 3.4))
for ax, (column, order, title) in zip(axes, [('wt_side', ['Left', 'Right'], 'Hemisphere'),
                                             ('wt_ap', ['Anterior', 'Posterior'], 'Front / back'),
                                             ('wt_si', ['Superior', 'Inferior'], 'Upper / lower')]):
    counts = ok[column].value_counts().reindex(order).fillna(0)
    plots.bar_panel(ax, order, counts.values, colors=[plots.PALETTE[0], plots.PALETTE[1]], title=title, ylabel='patients')
fig.suptitle('Tumor position relative to the brain centre', y=1.04)
fig.tight_layout()
plots.save_fig(fig, FIG, '12_tumor_side', data=ok[['case_id', 'wt_side', 'wt_ap', 'wt_si']])

In [ ]:
# 13-14. Intensity histograms before and after normalization
rng = np.random.default_rng(0)
histogram_figures = [
    ('raw_hist', stats['raw_bins'], '13_raw_intensity_histograms', True,
     'Raw intensities differ a lot between patients (scanner differences)'),
    ('norm_hist', stats['norm_bins'], '14_normalized_intensity_histograms', False,
     'After z-score normalization the patients line up'),
]
for key, edges, name, log_x, title in histogram_figures:
    histograms = stats[key].astype(float)                    # (patients, modalities, bins)
    centers = np.sqrt(edges[:-1] * edges[1:]) if log_x else (edges[:-1] + edges[1:]) / 2
    sample = rng.choice(len(histograms), size=min(30, len(histograms)), replace=False) if len(histograms) else []
    fig, axes = plt.subplots(1, 4, figsize=(17, 3.6))
    tables = []
    for c, modality in enumerate(modalities):
        ax = axes[c]
        for i in sample:
            ax.plot(centers, histograms[i, c] / max(histograms[i, c].sum(), 1), color=plots.MUTED, alpha=0.35,
                    linewidth=0.8)
        overall = histograms[:, c].sum(0) / max(histograms[:, c].sum(), 1)
        ax.plot(centers, overall, color=plots.MOD_COLORS[modality], linewidth=2.2, label='all patients')
        nonzero = np.where(overall > 1e-4)[0]
        if nonzero.size:
            ax.set_xlim(edges[max(nonzero[0] - 2, 0)], edges[min(nonzero[-1] + 3, len(edges) - 1)])
        if log_x:
            plots.log_axis(ax, 'x')
        ax.set_title(plots.MOD_TITLES[modality])
        ax.set_xlabel('intensity' + (' (log scale)' if log_x else ' (z-score)'))
        ax.set_ylabel('fraction of brain voxels')
        tables.append(pd.DataFrame({'modality': modality, 'bin_center': centers, 'fraction_all_patients': overall}))
    axes[0].legend(loc='upper right')
    fig.suptitle(title + ' (grey: 30 individual patients)', y=1.04)
    fig.tight_layout()
    plots.save_fig(fig, FIG, name, data=pd.concat(tables))

In [ ]:
# 15. Spread of raw intensity levels between patients
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
modality_titles = [plots.MOD_TITLES[m] for m in modalities]
modality_colors = [plots.MOD_COLORS[m] for m in modalities]
plots.box_panel(axes[0], [ok[f'raw_{m}_p50'] for m in modalities], modality_titles, colors=modality_colors,
                ylabel='median raw intensity', title='Median brain intensity per patient')
plots.box_panel(axes[1], [ok[f'raw_{m}_p99'] for m in modalities], modality_titles, colors=modality_colors,
                ylabel='99th percentile raw intensity', title='Bright end per patient')
for ax in axes:
    plots.log_axis(ax, 'y')
fig.tight_layout()
plots.save_fig(fig, FIG, '15_raw_intensity_spread',
               data=ok[['case_id'] + [f'raw_{m}_{q}' for m in modalities for q in ('p01', 'p50', 'p99')]])

In [ ]:
# 16. Correlation between tumor measurements
correlation_columns = {'brain_cm3': 'Brain', 'vol_wt_cm3': 'WT', 'vol_tc_cm3': 'TC', 'vol_et_cm3': 'ET',
                       'vol_ncr_cm3': 'NCR', 'vol_ed_cm3': 'ED', 'wt_n_components': 'Parts',
                       'wt_extent_x_mm': 'Extent x', 'wt_extent_y_mm': 'Extent y', 'wt_extent_z_mm': 'Extent z',
                       'n_tumor_slices': 'Tumor slices'}
correlation = ok[list(correlation_columns)].astype(float).corr(method='spearman')
fig, ax = plt.subplots(figsize=(8, 6.8))
image = ax.imshow(correlation.values, cmap=plots.DIV_CMAP, vmin=-1, vmax=1)
ax.grid(False)
ax.set_xticks(range(len(correlation_columns)))
ax.set_xticklabels(correlation_columns.values(), rotation=45, ha='right')
ax.set_yticks(range(len(correlation_columns)))
ax.set_yticklabels(correlation_columns.values())
for i in range(len(correlation)):
    for j in range(len(correlation)):
        value = correlation.values[i, j]
        if np.isfinite(value):
            ax.text(j, i, f'{value:.2f}', ha='center', va='center', fontsize=7,
                    color='white' if abs(value) > 0.6 else plots.INK)
fig.colorbar(image, ax=ax, shrink=0.8, label='Spearman correlation')
ax.set_title('How the tumor measurements relate to each other')
fig.tight_layout()
plots.save_fig(fig, FIG, '16_measurement_correlation', data=correlation.reset_index())

In [ ]:
# 17. Are train / val / test similar? (they should be)
split_order = ['train', 'val', 'test']
split_colors = [plots.SPLIT_COLORS[s] for s in split_order]
fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))
plots.box_panel(axes[0], [ok.loc[ok['split'] == s, 'vol_wt_cm3'] for s in split_order], split_order,
                colors=split_colors, ylabel='cm³', title='Whole-tumor volume')
plots.box_panel(axes[1], [ok.loc[ok['split'] == s, 'wt_brain_pct'] for s in split_order], split_order,
                colors=split_colors, ylabel='% of brain', title='Tumor share of brain')
counts = ok['split'].value_counts().reindex(split_order).fillna(0)
plots.bar_panel(axes[2], split_order, counts.values, colors=split_colors, title='Patients per split', ylabel='patients')
fig.suptitle('Are train / validation / test similar?', y=1.04)
fig.tight_layout()
plots.save_fig(fig, FIG, '17_split_balance', data=ok[['case_id', 'split', 'vol_wt_cm3', 'wt_brain_pct']])

In [ ]:
# 18-23. More distributions
distribution_figures = [
    ('18_tumor_parts', [('wt_n_components', 'Number of separate tumor parts', 'parts')]),
    ('19_tumor_extent', [('wt_extent_x_mm', 'Extent left-right', 'mm'), ('wt_extent_y_mm', 'Extent front-back', 'mm'),
                         ('wt_extent_z_mm', 'Extent up-down', 'mm')]),
    ('20_tumor_slices_per_patient', [('n_tumor_slices', 'Axial slices containing tumor', 'slices')]),
    ('21_brain_crop_size', [('crop_x', 'Crop size x', 'voxels'), ('crop_y', 'Crop size y', 'voxels'),
                            ('crop_z', 'Crop size z', 'voxels')]),
    ('22_storage_per_patient', [('npz_mb', 'Preprocessed file size', 'MB')]),
    ('23_largest_part_share', [('wt_largest_component_pct', 'Share of the tumor in its largest part', '%')]),
]
for name, panels in distribution_figures:
    fig, axes = plt.subplots(1, len(panels), figsize=(4.6 * len(panels), 3.4), squeeze=False)
    for i, (ax, (column, title, unit)) in enumerate(zip(axes.ravel(), panels)):
        plots.hist_panel(ax, ok[column], bins=30, color=plots.PALETTE[i], title=title, xlabel=unit)
    fig.tight_layout()
    plots.save_fig(fig, FIG, name, data=ok[['case_id'] + [column for column, _, _ in panels]])

print('\nAll dataset figures saved in', FIG)

## Done
Everything is on Drive:
- `data/preprocessed/` – the shards (what notebooks 02–04 use)
- `data/meta/` – `cases.csv`, `slices.csv`, `split.csv`, statistics
- `data/figures/` – all graphs (PNG + PDF + CSV)

The last cell makes sure every file has finished uploading to Drive.

In [ ]:
env.flush_drive()